# FPL Value: projection model playground

Everything the Projections page shows, as Python you can poke at. Run from the repo root or this folder:

```bash
pip install -r requirements.txt jupyter
jupyter notebook notebooks/projections.ipynb
```

Pull the repo first (`git pull`) so `data/current` has the latest weekly refresh.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / 'fplvalue').exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import fplvalue as fv
from fplvalue.simulate import captain_table
from fplvalue.optimise import optimise
import fplvalue.backtest as bt
pd.set_option('display.width', 200, 'display.max_columns', 40)

## 1. Load the data and project
`last` is last season's final snapshot: it's the prior for player rates, start rates and team strength.

In [ ]:
snap = fv.load()                      # data/current
last = fv.last_season(snap)           # e.g. data/history/2026-07-20
prm = fv.Params()                     # every tunable setting lives here
proj, fixtures_long = fv.project(fv.state_from_snapshot(snap, last), prm, detail=True)
gws = proj.attrs['gws']; print('projecting', gws, '· prior:', last.path.name if last else None)

In [ ]:
cols = ['web_name', 'team_short_name', 'pos', 'price', 'next_opp', 'avail_next', 'p_start', 'next_gw'] + [f'gw_{g}' for g in gws] + ['next_n']
proj[cols].head(20).round(2)

## 2. Look inside one player
The next-GW projection is the sum of these components; the inputs underneath are what drive them.

In [ ]:
name = 'Bogle'
row = proj[proj.web_name == name].iloc[0]
comp = row[[f'next_{c}' for c in fv.COMPONENTS]].rename(lambda c: c[5:])
display(comp.astype(float).round(2).to_frame('pts'))
row[['p_start', 'p_sub', 'm_start', 'q60', 'xg90', 'xa90', 'hit_rate', 'bonus90', 'avail_next']].astype(float).round(3)

In [ ]:
# per-fixture detail: fixture multipliers, xG for/against, clean-sheet odds
fixtures_long[fixtures_long.id == row.id][['event', 'opp', 'home', 'xgf', 'xga', 'mult_att', 'xmins', 'goals', 'assists', 'cs', 'xp']].round(2)

## 3. Best value by position
Projected points over the window per £m, among likely starters.

In [ ]:
v = proj[proj.p_start >= 0.6].assign(per_m=lambda d: d.next_n / d.price)
v.sort_values('per_m', ascending=False).groupby('pos').head(5).sort_values(['pos', 'per_m'], ascending=[True, False])[
    ['pos', 'web_name', 'team_short_name', 'price', 'next_n', 'per_m', 'selected_by_percent']].round(2)

## 4. Captaincy
Monte Carlo of the next gameweek: mean, 90th percentile, P(10+ points), P(≤2).

In [ ]:
captain_table(proj, fixtures_long, prm, top=30).head(12).round(2)

## 5. Squads and transfers
From scratch, or from your own team (your FPL team ID is in the URL of your points page).

In [ ]:
squad, info = optimise(proj, budget=100.0)
print(info)
squad[['web_name', 'team_short_name', 'pos', 'price', 'next_opp', 'next_gw', 'next_n', 'is_starter', 'is_captain']].round(2)

In [ ]:
TEAM_ID = None        # e.g. 1234567
FREE_TRANSFERS = 1
if TEAM_ID:
    from fplvalue.optimise import fetch_team
    ids, bank = fetch_team(TEAM_ID, snap.last_finished_gw)
    budget = round(bank + proj[proj.id.isin(ids)].price.sum(), 1)
    names = proj.set_index('id').web_name
    for k in range(0, 3):
        sq, inf = optimise(proj, budget, current=ids, free_transfers=FREE_TRANSFERS, max_transfers=k)
        print(k, 'transfers | XI value', inf['xi_value'], '| hits', inf['extra_transfers'],
              '| out', [names[i] for i in inf['out']], '| in', [names[i] for i in inf['in']])

## 6. Backtest
Rewind to before each deadline, project that gameweek, compare with what happened. `regulars` = started 2+ of their last 4.

In [ ]:
res = bt.run(prm, verbose=False)
for k, v in res.items():
    print(v['label']); display(v['regulars'].round(3))

In [ ]:
# calibration: are 5-point projections really worth 5 points?
bt.calibration(res['last_season']['rows']).round(2)

## 7. Tweak the model
Change a setting, re-run the backtest, see if it helps. Lower RMSE / higher spearman / higher top-20 is better.

In [ ]:
from dataclasses import replace
def score(p):
    rows = bt.season_rows(last, range(5, 39), p)
    m = bt.metrics(rows[rows.regular], cols=('model',)).iloc[0]
    return round(m.rmse, 3), round(m.spearman, 3), round(m.top20_actual, 2)

print('current ', score(prm))
for kw in [dict(k_min=1.5), dict(decay=0.9), dict(k_xg=12, k_xa=12), dict(cs_cal=0.9)]:
    print(kw, score(replace(prm, **kw)))

## Ideas to try next
- **Set pieces / penalties**: a per-player penalty share from `penalties_order` in players.csv, adding pen xG to takers.
- **Bonus from BPS**: model BPS per 90 and convert to bonus via rank within the match.
- **Minutes from news**: parse "rotation" / "late fitness test" phrases.
- **Longer horizon**: evaluate the 5-week total, not just next week, by rewinding and comparing with the sum of the next five.
- **Machine learning**: once a season of weekly `projections.csv` snapshots exists in `data/history`, fit a residual model on top.